In [19]:
import pandas as pd
import dask.dataframe as dd
from datetime import datetime
import json

In [20]:
# Main processing
longitude_interval_ar = 0.625
latitude_interval_ar = 0.5
hours_multiple_ar = 3

longitude_interval_precip = 0.1
latitude_interval_precip = 0.1
hours_multiple_precip = 0.5  # For 30 minutes

In [21]:
yyyy = 2005

In [22]:
zwd_file_path = f"/root/data/rrr/integrated_weather_dataset/data/processed/Troposphere/final/{yyyy}.csv"
zwd_df = pd.read_csv(zwd_file_path)
zwd_df['Timestamp'] = zwd_df['Timestamp'].astype(str)
zwd_df['Timestamp'] = pd.to_datetime(zwd_df['Timestamp'], errors='coerce')
zwd_df = zwd_df.dropna()

In [23]:
# Read AR catalog data (Rutz)
rutz_file_path = f"/root/data/rrr/integrated_weather_dataset/data/processed/Rutz/{yyyy}.csv"
rutz_df = pd.read_csv(rutz_file_path)
rutz_df = rutz_df.rename(columns={"longitude": "Longitude"})
rutz_df = rutz_df.rename(columns={"latitude": "Latitude"})
rutz_df['Timestamp'] = pd.to_datetime(rutz_df['Timestamp'], errors='coerce')
rutz_df = rutz_df.rename(columns={f'ARs': f'Label'})

In [24]:
# Read AR catalog data (Guan)
guan_file_path = f"/root/data/rrr/integrated_weather_dataset/data/processed/Guan/{yyyy}.csv"
guan_df = pd.read_csv(guan_file_path)
guan_df['Timestamp'] = pd.to_datetime(guan_df['Timestamp'], errors='coerce')
guan_df = guan_df.rename(columns={f'Guan_AR_Label': f'Label'})

In [25]:
# Read Precipitation data
precip_file_path = f"/root/data/rrr/integrated_weather_dataset/data/processed/Precipitation/{yyyy}.csv"
precip_df = pd.read_csv(precip_file_path)
precip_df['Timestamp'] = pd.to_datetime(precip_df['Timestamp'], errors='coerce')
precip_df = precip_df.rename(columns={f'Precipitation': f'Label'})

In [26]:
# Helper function to round timestamps to the nearest specified multiple of hours
def round_time_to_nearest_multiple(dt, hours_multiple):
    """Round a datetime to the nearest multiple of hours."""
    multiple_seconds = hours_multiple * 3600
    seconds_since_midnight = dt.hour * 3600 + dt.minute * 60 + dt.second
    nearest_multiple = round(seconds_since_midnight / multiple_seconds) * multiple_seconds
    total_seconds = dt.replace(hour=0, minute=0, second=0, microsecond=0).timestamp() + nearest_multiple
    return datetime.fromtimestamp(total_seconds)

In [27]:
def apply_rounding(df, hours_multiple, grid_lons_lookup, grid_lats_lookup):
    df['Rounded Timestamp'] = df['Timestamp'].apply(lambda x: round_time_to_nearest_multiple(x, hours_multiple))
    df['Rounded Longitude'] = df['Longitude'].apply(lambda x: grid_lons_lookup.get(int(round(x * 1000)), None))
    df['Rounded Latitude'] = df['Latitude'].apply(lambda x: grid_lats_lookup.get(int(round(x * 1000)), None))
    return df

In [28]:
# Helper function to create lookup grids for rounding coordinates
def create_grid_lookup(grid_data, interval):
    """Create a lookup dictionary for coordinate rounding."""
    interval_threshold = interval / 2
    grid_lookup = {}
    for item in grid_data:
        lower = item - interval_threshold
        upper = item + interval_threshold
        for key in range(int(lower * 1000), int(upper * 1000) + 1):
            grid_lookup[key] = item
    return grid_lookup

In [29]:

def merge_zwd_with_data(zwd_df, data_df, longitude_interval, latitude_interval, hours_multiple, name = 'Rutz'):
    """
    Merges ZWD data with another dataset (e.g., AR catalogs) by creating a grid lookup
    for spatial rounding and timestamp rounding. Includes both exact match and label assignment.

    Parameters:
        zwd_df (pd.DataFrame): ZWD dataset with 'Timestamp', 'Longitude', and 'Latitude'.
        data_df (pd.DataFrame): Dataset to merge with, including 'time', 'lon', 'lat', and 'label'.
        longitude_interval (float): Interval for longitude rounding.
        latitude_interval (float): Interval for latitude rounding.
        hours_multiple (int): Interval for timestamp rounding in hours.

    Returns:
        pd.DataFrame: ZWD dataset merged with the other dataset, including exact match and label columns.
    """
    
    
    # Create grid lookups
    grid_lons_lookup = create_grid_lookup(data_df['Longitude'].unique(), longitude_interval)
    grid_lats_lookup = create_grid_lookup(data_df['Latitude'].unique(), latitude_interval)

    # Apply rounding
    zwd_df = apply_rounding(zwd_df, hours_multiple, grid_lons_lookup, grid_lats_lookup)
    
    # Rename columns for merging
    data_df = data_df.rename(columns={
        'Latitude': 'Rounded Latitude',
        'Longitude': 'Rounded Longitude',
        'Label': f'{name}_Label'  # Rename 'label' to a specific column for clarity
    })
    # Merge for exact match

    merged_df = pd.merge(
        zwd_df,
        data_df,
        on=['Timestamp', 'Rounded Longitude', 'Rounded Latitude'],  # Exact match on original coordinates and timestamp
        how='left'
    )

    # Rename columns for merging
    data_df = data_df.rename(columns={
        'Timestamp': 'Rounded Timestamp',
    })

    # Merge for approximate match (spatial and timestamp rounding)
    merged_df = pd.merge(
        merged_df,
        data_df,
        on=['Rounded Timestamp', 'Rounded Latitude', 'Rounded Longitude'],
        how='left',
        suffixes=('', '_approx')  # To differentiate between exact match and approximate
    )

    # Assign approximate match label
    # merged_df[f'{name}_Label_approx'] = merged_df[f'{name}_Label_approx'].fillna(0).astype(int)

    # Clean up intermediate columns if necessary
    merged_df.drop(
        columns=['Rounded Timestamp', 'Rounded Latitude', 'Rounded Longitude'],
        inplace=True,
        errors='ignore'
    )
    
    merged_df = merged_df.rename(columns={
        f'{name}_Label': f'{name}_Label_exact',
    })

    return merged_df

In [30]:
zwd_with_rutz = merge_zwd_with_data(zwd_df, rutz_df, longitude_interval_ar, latitude_interval_ar, hours_multiple_ar, name = 'Rutz')
zwd_with_rutz

,Timestamp,Site,Latitude,Longitude,ZWD,Wet Gradient North,Wet Gradient East,IVT,Rutz_Label_exact,IVT_approx,Rutz_Label_approx
0,2005-01-01 00:00:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,92.531807,0.0,92.531807,0.0
1,2005-01-01 00:05:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,NaN,NaN,92.531807,0.0
2,2005-01-01 00:10:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,NaN,NaN,92.531807,0.0
3,2005-01-01 00:15:00,7ODM,34.116407,-117.093192,10.1,-2.89,-2.09,NaN,NaN,92.531807,0.0
4,2005-01-01 00:20:00,7ODM,34.116407,-117.093192,10.0,-2.89,-2.09,NaN,NaN,92.531807,0.0
...,...,...,...,...,...,...,...,...,...,...,...
30047965,2005-12-31 23:35:00,WWMT,33.955313,-116.653855,100.2,-1.87,-1.10,NaN,NaN,NaN,NaN
30047966,2005-12-31 23:40:00,WWMT,33.955313,-116.653855,99.1,-1.93,-1.08,NaN,NaN,NaN,NaN
30047967,2005-12-31 23:45:00,WWMT,33.955313,-116.653855,97.8,-2.01,-1.05,NaN,NaN,NaN,NaN
30047968,2005-12-31 23:50:00,WWMT,33.955313,-116.653855,96.4,-2.08,-1.02,NaN,NaN,NaN,NaN


In [31]:
zwd_rutz_guan = merge_zwd_with_data(zwd_with_rutz, guan_df, longitude_interval_ar, latitude_interval_ar, hours_multiple_ar, name = 'Guan')
zwd_rutz_guan

,Timestamp,Site,Latitude,Longitude,ZWD,Wet Gradient North,Wet Gradient East,IVT,Rutz_Label_exact,IVT_approx,Rutz_Label_approx,Guan_Label_exact,Guan_Label_approx
0,2005-01-01 00:00:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,92.531807,0.0,92.531807,0.0,0.0,0.0
1,2005-01-01 00:05:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,NaN,NaN,92.531807,0.0,NaN,0.0
2,2005-01-01 00:10:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,NaN,NaN,92.531807,0.0,NaN,0.0
3,2005-01-01 00:15:00,7ODM,34.116407,-117.093192,10.1,-2.89,-2.09,NaN,NaN,92.531807,0.0,NaN,0.0
4,2005-01-01 00:20:00,7ODM,34.116407,-117.093192,10.0,-2.89,-2.09,NaN,NaN,92.531807,0.0,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
30047965,2005-12-31 23:35:00,WWMT,33.955313,-116.653855,100.2,-1.87,-1.10,NaN,NaN,NaN,NaN,NaN,NaN
30047966,2005-12-31 23:40:00,WWMT,33.955313,-116.653855,99.1,-1.93,-1.08,NaN,NaN,NaN,NaN,NaN,NaN
30047967,2005-12-31 23:45:00,WWMT,33.955313,-116.653855,97.8,-2.01,-1.05,NaN,NaN,NaN,NaN,NaN,NaN
30047968,2005-12-31 23:50:00,WWMT,33.955313,-116.653855,96.4,-2.08,-1.02,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
zwd_rutz_guan_precip = merge_zwd_with_data(zwd_rutz_guan, precip_df, longitude_interval_precip, latitude_interval_precip, hours_multiple_precip, name = "Precipitation")
zwd_rutz_guan_precip

,Timestamp,Site,Latitude,Longitude,ZWD,Wet Gradient North,Wet Gradient East,IVT,Rutz_Label_exact,IVT_approx,Rutz_Label_approx,Guan_Label_exact,Guan_Label_approx,Precipitation_Label_exact,Precipitation_Label_approx
0,2005-01-01 00:00:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,92.531807,0.0,92.531807,0.0,0.0,0.0,0.0,0.00
1,2005-01-01 00:00:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,92.531807,0.0,92.531807,0.0,0.0,0.0,0.0,0.00
2,2005-01-01 00:00:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,92.531807,0.0,92.531807,0.0,0.0,0.0,0.0,0.00
3,2005-01-01 00:00:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,92.531807,0.0,92.531807,0.0,0.0,0.0,0.0,0.00
4,2005-01-01 00:05:00,7ODM,34.116407,-117.093192,10.2,-2.89,-2.09,NaN,NaN,92.531807,0.0,NaN,0.0,NaN,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30049557,2005-12-31 23:35:00,WWMT,33.955313,-116.653855,100.2,-1.87,-1.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.32
30049558,2005-12-31 23:40:00,WWMT,33.955313,-116.653855,99.1,-1.93,-1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.32
30049559,2005-12-31 23:45:00,WWMT,33.955313,-116.653855,97.8,-2.01,-1.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30049560,2005-12-31 23:50:00,WWMT,33.955313,-116.653855,96.4,-2.08,-1.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
import time
def merge_with_ffw(trop_df, ffw_data):
    trop_df['ffw'] = 0
    trop_df['Timestamp'] = pd.to_datetime(trop_df['Timestamp'])
    # Process each flash flood warning entry
    for ffw in ffw_data:
        t1 = time.time()
        begin = pd.to_datetime(ffw['begin']).tz_localize(None)
        sites = ffw['sites']
        if begin.year != yyyy or len(sites)==0:
            continue
        end = pd.to_datetime(ffw['end']).tz_localize(None)
        

        # Filter rows matching the time range and sites
        mask = (trop_df['Timestamp'] >= begin) & (trop_df['Timestamp'] <= end)
        if sites:
            mask &= trop_df['Site'].isin(sites)

        # Set 'ffw' column for matching rows
        trop_df.loc[mask, 'ffw'] = 1
        print(begin, end, time.time()-t1)
    return trop_df

In [14]:
ffw_file_path = f"/root/data/rrr/integrated_weather_dataset/data/processed/Flash_Flood/data.json"
with open(ffw_file_path, 'r') as f:
    ffw_data = json.load(f)

In [15]:
for entry in ffw_data:
    entry['begin_dt'] = pd.to_datetime(entry['begin'])

# Filter entries with year 2005
filtered_data = [entry for entry in ffw_data if entry['begin_dt'].year == 2005 and len(entry['sites'])>0]

# Sort the filtered entries by 'begin' timestamp
sorted_data = sorted(filtered_data, key=lambda x: x['begin_dt'])

# Remove the temporary 'begin_dt' key before returning the result
for entry in sorted_data:
    del entry['begin_dt']

# Print the sorted and filtered data
print(sorted_data)

[{'begin': '2005-01-03T08:36:00Z', 'end': '2005-01-03T14:30:00Z', 'sites': ['AOA1', 'CBHS', 'CIRX', 'CSN1', 'FMTP', 'FMVT', 'KBRC', 'LAPC', 'MPWD', 'OAT2', 'ROCK', 'SFDM', 'SOMT', 'SPK1', 'TOST']}, {'begin': '2005-01-03T08:36:00Z', 'end': '2005-01-03T14:30:00Z', 'sites': ['AOA1', 'CBHS', 'CIRX', 'CSN1', 'FMTP', 'FMVT', 'KBRC', 'LAPC', 'MPWD', 'OAT2', 'ROCK', 'SFDM', 'SOMT', 'SPK1', 'TOST']}, {'begin': '2005-01-03T08:36:00Z', 'end': '2005-01-03T14:30:00Z', 'sites': ['AOA1', 'CBHS', 'CIRX', 'CSN1', 'FMTP', 'FMVT', 'KBRC', 'LAPC', 'MPWD', 'OAT2', 'ROCK', 'SFDM', 'SOMT', 'SPK1', 'TOST']}, {'begin': '2005-01-03T08:36:00Z', 'end': '2005-01-03T14:30:00Z', 'sites': ['AOA1', 'CBHS', 'CIRX', 'CSN1', 'FMTP', 'FMVT', 'KBRC', 'LAPC', 'MPWD', 'OAT2', 'ROCK', 'SFDM', 'SOMT', 'SPK1', 'TOST']}, {'begin': '2005-01-03T08:36:00Z', 'end': '2005-01-03T14:30:00Z', 'sites': ['AOA1', 'CBHS', 'CIRX', 'CSN1', 'FMTP', 'FMVT', 'KBRC', 'LAPC', 'MPWD', 'OAT2', 'ROCK', 'SFDM', 'SOMT', 'SPK1', 'TOST']}, {'begin': '200

In [16]:
len(sorted_data)

10374

In [18]:
final_df = merge_with_ffw(zwd_df, sorted_data)

2005-01-03 08:36:00 2005-01-03 14:30:00 0.9608240127563477
2005-01-03 08:36:00 2005-01-03 14:30:00 0.9075527191162109
2005-01-03 08:36:00 2005-01-03 14:30:00 0.8855335712432861
2005-01-03 08:36:00 2005-01-03 14:30:00 0.9063286781311035
2005-01-03 08:36:00 2005-01-03 14:30:00 0.892503023147583
2005-01-03 08:36:00 2005-01-03 14:30:00 0.9073634147644043
2005-01-03 08:36:00 2005-01-03 14:30:00 0.9176731109619141
2005-01-03 08:36:00 2005-01-03 14:30:00 0.8852424621582031
2005-01-03 08:36:00 2005-01-03 14:30:00 0.8805360794067383


KeyboardInterrupt: 

In [9]:
from collections import Counter

years = [pd.to_datetime(entry['begin']).year for entry in sorted_ffw_data]

# Count occurrences of each year
year_counts = Counter(years)

# Output unique years and their counts
print("Unique Years:", list(year_counts.keys()))
print("Counts per Year:", year_counts)

Unique Years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Counts per Year: Counter({2005: 11970, 2008: 7522, 2023: 7474, 2017: 7065, 2022: 6707, 2015: 6678, 2019: 6565, 2013: 6254, 2014: 4644, 2006: 4619, 2010: 4409, 2007: 4076, 2012: 4046, 2021: 3957, 2011: 3824, 2018: 3769, 2016: 3198, 2020: 2694, 2009: 1160})


In [26]:
final_df = merge_with_ffw(zwd_rutz_guan_precip, ffw_data)

here
2005-02-21 21:50:00 2005-02-22 00:45:00
2005-05-06 01:43:00 2005-05-06 02:45:00
2005-05-06 02:45:00 2005-05-06 04:45:00
2005-07-23 22:37:00 2005-07-23 23:30:00
2005-07-23 23:47:00 2005-07-24 00:45:00
2005-07-24 00:42:00 2005-07-24 01:45:00
2005-07-24 20:30:00 2005-07-24 21:30:00
2005-07-24 21:26:00 2005-07-24 23:30:00
2005-07-24 22:14:00 2005-07-25 00:15:00
2005-07-24 23:17:00 2005-07-25 01:15:00
2005-07-25 01:00:00 2005-07-25 01:45:00
2005-08-16 01:25:00 2005-08-16 03:30:00
2005-10-18 00:33:00 2005-10-18 03:30:00
2005-10-18 03:18:00 2005-10-18 04:45:00
2005-10-18 11:18:00 2005-10-18 13:15:00
2005-10-18 12:30:00 2005-10-18 14:30:00
2005-10-18 14:28:00 2005-10-18 17:30:00
2005-02-21 21:50:00 2005-02-22 00:45:00
2005-05-06 01:43:00 2005-05-06 02:45:00
2005-05-06 02:45:00 2005-05-06 04:45:00
2005-07-23 22:37:00 2005-07-23 23:30:00
2005-07-23 23:47:00 2005-07-24 00:45:00
2005-07-24 00:42:00 2005-07-24 01:45:00
2005-07-24 20:30:00 2005-07-24 21:30:00
2005-07-24 21:26:00 2005-07-24 23:3

KeyboardInterrupt: 

In [ ]:
final_df.to_csv('tmp.csv')

In [ ]:
# # Merge ZWD dataset with AR catalogs, precipitation, and flash flood datasets
# def merge_datasets(zwd_df, rutz_df, guan_df, precip_df, ffw_data, longitude_interval, latitude_interval, ar_hours, precip_minutes):
#     """
#     Merge ZWD dataset with multiple datasets including AR catalogs, precipitation,
#     and flash flood data.
#     """
#     # Initialize output DataFrame
#     output_df = zwd_df.copy()
#     output_df['Rutz_exact_match_label'] = 0
#     output_df['Rutz_Label'] = 0
#     output_df['Guan_exact_match_label'] = 0
#     output_df['Guan_Label'] = 0
#     output_df['Precipitation_exact_match'] = None
#     output_df['Precipitation'] = None
#     output_df['Flash_flood_label'] = 0

#     # Create grid lookup for Rutz AR catalog
#     rutz_grid_lons_lookup = create_grid_lookup(rutz_df['lon'].unique(), longitude_interval)
#     rutz_grid_lats_lookup = create_grid_lookup(rutz_df['lat'].unique(), latitude_interval)

#     # Round ZWD timestamps and coordinates for Rutz AR catalog
#     zwd_df['Rounded Timestamp'] = zwd_df['Timestamp'].apply(lambda x: round_time_to_nearest_multiple(x, ar_hours))
#     zwd_df['Rounded Longitude'] = zwd_df['Longitude'].apply(lambda x: rutz_grid_lons_lookup.get(int(round(x * 1000)), None))
#     zwd_df['Rounded Latitude'] = zwd_df['Latitude'].apply(lambda x: rutz_grid_lats_lookup.get(int(round(x * 1000)), None))

#     # Merge with Rutz AR catalog
#     rutz_df = rutz_df.rename(columns={
#         'time': 'Rounded Timestamp',
#         'lon': 'Rounded Longitude',
#         'lat': 'Rounded Latitude',
#         'label': 'Rutz_Label'
#     })
#     output_df = pd.merge(zwd_df, rutz_df, on=['Rounded Timestamp', 'Rounded Longitude', 'Rounded Latitude'], how='left')
#     output_df['Rutz_Label'] = output_df['Rutz_Label'].fillna(0).astype(int)

#     # Repeat for Guan AR catalog
#     guan_grid_lons_lookup = create_grid_lookup(guan_df['lon'].unique(), longitude_interval)
#     guan_grid_lats_lookup = create_grid_lookup(guan_df['lat'].unique(), latitude_interval)
#     zwd_df['Rounded Longitude'] = zwd_df['Longitude'].apply(lambda x: guan_grid_lons_lookup.get(int(round(x * 1000)), None))
#     zwd_df['Rounded Latitude'] = zwd_df['Latitude'].apply(lambda x: guan_grid_lats_lookup.get(int(round(x * 1000)), None))
#     guan_df = guan_df.rename(columns={
#         'time': 'Rounded Timestamp',
#         'lon': 'Rounded Longitude',
#         'lat': 'Rounded Latitude',
#         'label': 'Guan_Label'
#     })
#     output_df = pd.merge(output_df, guan_df, on=['Rounded Timestamp', 'Rounded Longitude', 'Rounded Latitude'], how='left')
#     output_df['Guan_Label'] = output_df['Guan_Label'].fillna(0).astype(int)

#     # Merge with Precipitation dataset
#     precip_df['Rounded Timestamp'] = precip_df['Timestamp'].apply(lambda x: round_time_to_nearest_multiple(x, precip_minutes / 60))
#     output_df = pd.merge(output_df, precip_df, on=['Rounded Timestamp', 'Longitude', 'Latitude'], how='left')
#     output_df['Precipitation'] = output_df['Precipitation'].fillna(0)

#     # Integrate Flash Flood dataset
#     output_df['Flash_flood_label'] = 0
#     for ffw in ffw_data:
#         begin = pd.to_datetime(ffw['begin']).tz_localize(None)
#         end = pd.to_datetime(ffw['end']).tz_localize(None)
#         sites = ffw['sites']
#         mask = (output_df['Timestamp'] >= begin) & (output_df['Timestamp'] <= end) & output_df['Site'].isin(sites)
#         output_df.loc[mask, 'Flash_flood_label'] = 1

#     # Drop intermediate columns
#     output_df = output_df.drop(columns=["Rounded Timestamp", "Rounded Longitude", "Rounded Latitude"])

#     return output_df